In [198]:
import numpy as np 
import pandas as pd 
import MLM.angle_between as angle
import MLM.moire_lattice_vector_finder as mlf 
import MLM.structure_writer as sw

In [199]:
file_1 = "../moire_structures/mos2/mos2_monolayer.vasp"
file_2 = "../moire_structures/mos2/mos2_monolayer.vasp"
file_3 = "../moire_structures/mos2/mos2_monolayer.vasp"

In [200]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)

lattice_vectors3, atom_type_arr3, dat3 = mlf.read_vasp(file_3)


In [201]:
candidate = pd.read_pickle('mos2_60_60_ttl.pkl')

candidate.head()

,Theta,Phi,A1,A2,delvec,delang,norm
0,52.07,54.8,"[2808.2417397934, 1217.3752380791998]","[349.84258288279966, 3040.6962590762]",0.000815,0.000007,5301.383731
1,52.07,54.8,"[2808.2417397934, 1217.3752380791998]","[2458.3991569106, -1823.321020997]",0.000815,0.000005,5301.384407
2,52.07,54.8,"[-2808.2417397934, -1217.3752380791998]","[-349.84258288279966, -3040.6962590762]",0.000815,0.000007,5301.383731
3,52.07,54.8,"[-2808.2417397934, -1217.3752380791998]","[-2458.3991569106, 1823.321020997]",0.000815,0.000005,5301.384407
4,52.07,54.8,"[-349.84258288279966, -3040.6962590762]","[-2808.2417397934, -1217.3752380791998]",0.001375,0.000007,5301.383731


In [203]:
# Sort the DataFrame by 'degree' in ascending order and 'norm' in descending order, followed by 'vec_del' and 'delta_angle'
candidate_sorted = candidate.sort_values(by=['Theta','Phi', 'delvec', 'delang', 'norm'], ascending=[True, True, True, True, True])
final_df = candidate_sorted.groupby(['Theta', 'Phi'], as_index=False).first()


In [204]:
final_df.sort_values(by="norm", ascending=True).head(50)

,Theta,Phi,A1,A2,delvec,delang,norm
16266,60.00,120.00,"[9.4979996682, 0.0]","[4.7489987388, 8.2255083654]",0.000001,0.000004,16.451017
16265,60.00,119.99,"[4.7489987388, 8.2255083654]","[9.4979996682, 0.0]",0.001657,0.000004,16.451017
15960,59.99,119.99,"[4.7489987388, 8.2255083654]","[9.4979996682, 0.0]",0.001657,0.000004,16.451017
9827,38.22,60.01,"[-4.7490016596, 13.709180609]","[9.4979982078, 10.967344487199998]",0.002531,0.000006,25.129343
9499,38.21,60.01,"[-4.7490016596, 13.709180609]","[9.4979982078, 10.9673444872]",0.002531,0.000006,25.129343
10156,38.23,60.01,"[-4.7490016596, 13.709180609]","[9.4979982078, 10.967344487199998]",0.002531,0.000006,25.129343
10036,38.22,98.22,"[-4.7490016596, 13.709180609]","[9.4979982078, 10.967344487199998]",0.001717,0.000006,25.129343
15543,59.98,98.22,"[-4.749001659599999, 13.709180609]","[9.497998207799998, 10.9673444872]",0.001717,0.000006,25.129343
9172,38.20,60.01,"[-4.7490016596, 13.709180609]","[9.4979982078, 10.9673444872]",0.002531,0.000006,25.129343
15847,59.99,98.22,"[-4.749001659599999, 13.709180609]","[9.497998207799998, 10.9673444872]",0.001717,0.000006,25.129343


In [190]:

def filter_similar_rows(group):
    """
    For a given Theta group, keep rows that are distinct by the following rule:
    - A row is considered duplicate if there is a previously accepted row for which:
         abs(Phi_difference) < 0.1  and abs(norm_difference) < 1.
    """
    # Sort the group by Phi; you might also sort by norm as a tiebreaker if desired.
    group_sorted = group.sort_values(by='Phi')
    
    accepted_rows = []      # To store the accepted rows (as Series)
    accepted_indices = []   # To store their corresponding indices
    for idx, row in group_sorted.iterrows():
        duplicate = False
        # Compare with each already accepted row
        for accepted_row in accepted_rows:
            if (abs(row['Phi'] - accepted_row['Phi']) < 0.1) and (abs(row['norm'] - accepted_row['norm']) < 1):
                duplicate = True
                break
        if not duplicate:
            accepted_rows.append(row)
            accepted_indices.append(idx)
    return group_sorted.loc[accepted_indices]

# Assuming your DataFrame is called resdf
filtered_df = candidate.groupby('Theta', group_keys=False).apply(filter_similar_rows)



/tmp/ipykernel_39166/495652969.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_df = candidate.groupby('Theta', group_keys=False).apply(filter_similar_rows)


In [197]:
filtered_df.sort_values(by ='norm',ascending=True).head(50)

,Theta,Phi,A1,A2,delvec,delang,norm
34468,21.77,21.78,"[14.2469991372, 2.7418361217999996]","[-4.7489980086, -13.709180608999999]",0.001719,5.994542e-06,14.508434
46807,27.79,27.79,"[18.9960000666, -5.483672243599999]","[-14.246997676800003, -13.709180609]",0.001991,2.935767e-07,19.771662
19423,13.17,13.17,"[23.7449988054, 2.7418361218000005]","[-9.4979967474, -21.9346889744]",0.001481,5.220102e-06,23.902775
43808,27.79,6.01,"[-1.5830039607999993, 30.1601973398]","[26.9110008854, -13.709180608999999]",0.000536,4.485381e-06,30.201712
46138,27.79,21.79,"[17.412996105800005, 24.6765250962]","[-30.076999314400002, 2.7418361217999996]",0.001691,2.557022e-06,30.201714
35349,21.77,27.79,"[30.076999314400002, -2.7418361218000005]","[-17.4129961058, -24.6765250962]",0.003041,2.557022e-06,30.201714
27064,17.89,17.90,"[-4.749003850200001, 30.1601973398]","[28.494000465000003, -10.9673444872]",0.001837,5.660524e-06,30.531794
20800,13.17,21.79,"[7.9149949772, 35.643869583400004]","[26.9110023458, -24.676525096200002]",0.002044,6.595630e-07,36.512090
33026,21.77,13.17,"[26.9110023458, -24.6765250962]","[7.9149949772, 35.643869583400004]",0.002262,6.595630e-07,36.512090
32523,21.77,8.61,"[36.408998363, 2.7418361217999987]","[-15.829995065800002, -32.9020334616]",0.002063,4.761177e-06,36.512092


In [229]:
# pick you candidate
pick = final_df.loc[4634]

pick

Theta                                        21.77
Phi                                          38.21
A1                       [22.161999225800002, 0.0]
A2        [11.080997057200001, 19.192852852599998]
delvec                                     0.00124
delang                                    0.000004
norm                                     38.385706
Name: 4634, dtype: object

In [230]:
A1=np.array(pick['A1']	)
A2=np.array(pick['A2'] )
norm = pick['norm']

In [231]:
# Replication time 

replicate = int((norm/np.linalg.norm(lattice_vectors1['a']+lattice_vectors1['b']))+100)
replicate

112

In [232]:
new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                             b = lattice_vectors1['b'],
                                                                                             c = lattice_vectors1['c'],
                                                                                             atom_data = dat1,
                                                                                             atom_type_arr = atom_type_arr1,
                                                                                             natoms = len(dat1),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

In [233]:
new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                             b = lattice_vectors2['b'],
                                                                                             c = lattice_vectors2['c'],
                                                                                             atom_data = dat2,
                                                                                             atom_type_arr = atom_type_arr2,
                                                                                             natoms = len(dat2),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

In [234]:
new_lattice_vectors3, new_total_atoms3, replicated_atom3, replicated_type_arr3 = sw.replicate_atoms(a = lattice_vectors3['a'],
                                                                                             b = lattice_vectors3['b'],
                                                                                             c = lattice_vectors3['c'],
                                                                                             atom_data = dat3,
                                                                                             atom_type_arr = atom_type_arr3,
                                                                                             natoms = len(dat3),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

In [235]:
# Create a DataFrame for positions
positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

# Add the atom types to the DataFrame
positions1_df['type'] = replicated_type_arr1


In [236]:
# Create a DataFrame for positions
positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

# Add the atom types to the DataFrame
positions2_df['type'] = replicated_type_arr2 
positions2_df['z'] = positions2_df['z'] + 10


In [237]:
# Create a DataFrame for positions
positions3_df = pd.DataFrame(replicated_atom3, columns=['x', 'y', 'z'])

# Add the atom types to the DataFrame
positions3_df['type'] = replicated_type_arr3
positions3_df['z'] = positions3_df['z'] + 10*2 


In [238]:
theta1 = pick['Theta']

print(f"Twist Angle: {theta1}")

theta_rad = np.deg2rad(theta1)

print(f"Twist Angle in Radians: {theta_rad}")

rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                            [np.sin(theta_rad), np.cos(theta_rad), 0],
                            [0, 0, 1]])



# Rotate the positions in the upper layer
middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
rotated_positions = middle_layer_positions @ rotation_matrix.T  # Apply rotation

# Create a new DataFrame for the rotated positions
rotated_middle_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

# Add the atom types back to the rotated DataFrame
rotated_middle_layer['type'] = positions2_df['type'].values

Twist Angle: 21.77
Twist Angle in Radians: 0.37995817815916555


In [239]:
theta2 =   pick['Phi']
print(f"Twist Angle: {theta2}")

theta_rad = np.deg2rad(theta2)

rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                            [np.sin(theta_rad), np.cos(theta_rad), 0],
                            [0, 0, 1]])



# Rotate the positions in the upper layer
upper_layer_positions = positions3_df[['x', 'y', 'z']].values  # Extract x, y, z positions
rotated_positions = upper_layer_positions @ rotation_matrix.T  # Apply rotation

# Create a new DataFrame for the rotated positions
rotated_upper_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

# Add the atom types back to the rotated DataFrame
rotated_upper_layer['type'] = positions3_df['type'].values

Twist Angle: 38.21000000000241


In [240]:
# Concatenate DataFrames vertically
concat_vertical = pd.concat([positions1_df, rotated_middle_layer, rotated_upper_layer], ignore_index=True)

In [241]:
A1 = np.array(pick['A1'])
A2 = np.array(pick['A2'])
A3 = np.array([0, 0, 50.0])

print(f"A1:{A1}, A2:{A2}, A3:{A3}")
print(f"Len_A1: {np.linalg.norm(A1)}, Len_A2: {np.linalg.norm(A2)}")
print(f" Angle between A1 and A2: {angle.angle_between(A1,A2)}")

A1:[22.16199923  0.        ], A2:[11.08099706 19.19285285], A3:[ 0.  0. 50.]
Len_A1: 22.161999225800002, Len_A2: 22.16199667004828
 Angle between A1 and A2: 60.000003814645595


In [242]:
print("A1",A1)
print("A2",A2)
print("A1+A2",A1+A2)

A1 [22.16199923  0.        ]
A2 [11.08099706 19.19285285]
A1+A2 [33.24299628 19.19285285]


In [243]:
vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]
vertex 

[[0.0, 0.0],
 [np.float64(22.161999225800002), np.float64(0.0)],
 [np.float64(33.242996283000004), np.float64(19.192852852599998)],
 [np.float64(11.080997057200001), np.float64(19.192852852599998)]]

In [253]:
from matplotlib.path import Path

# Define the polygon points
polygon_points = vertex

# Create a Path object for the polygon
polygon_path = Path(polygon_points)


# Function to select atoms within the polygon with a tolerance
def select_atoms_in_polygon_path(df, polygon_path, tolerance=0.0):
    """
    Selects rows from a DataFrame where the 'x' and 'y' coordinates
    are inside or within a given tolerance of the specified polygon path.
    """
    xy_points = df[['x','y']].values  # Extract x, y coordinates
    # Add the radius parameter to include points near the boundary
    inside_mask = polygon_path.contains_points(xy_points, radius=tolerance)
    return df[inside_mask]

# Example usage:
selected_atoms_df = select_atoms_in_polygon_path(concat_vertical, polygon_path, tolerance=0.0)

In [254]:
selected_atoms_df['type'].unique()

array(['Mo', 'S'], dtype=object)

In [255]:
unique_types = selected_atoms_df['type'].unique()

sort_order = {value: index for index, value in enumerate(unique_types)}

print(f"Sort Order of Elements: {sort_order}")

# Create a new column to map the sort order
selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)

# Sort the DataFrame by the new sort key and drop the sort_key column
sorted_df = selected_atoms_df.sort_values(by='sort_key').drop(columns='sort_key').copy()


Sort Order of Elements: {'Mo': 0, 'S': 1}


/tmp/ipykernel_39166/3018769804.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)


In [256]:
sorted_df.head()

,x,y,z,type
75600,1.345550,0.502670,6.136667,Mo
76272,4.511550,0.502670,6.136667,Mo
383370,31.502734,18.459933,26.136667,Mo
382704,23.135470,18.852092,26.136667,Mo
76275,2.928549,3.244506,6.136667,Mo


In [257]:
for keys in sort_order:
    print(keys)
    print(len(sorted_df[sorted_df['type']==keys]))

Mo
147
S
294


In [258]:
element_count = " ".join(f"{len(sorted_df[sorted_df['type']==key])}" for key, _ in sort_order.items())
print(element_count)

147 294


In [259]:
A3

array([ 0.,  0., 50.])

In [260]:
def write_vasp_optimized(filename, A1, A2, A3, atom_data, sort_order):
    # Buffer for all the lines to write
    element_type = " ".join(f"{key}" for key, _ in sort_order.items())
    element_count = " ".join(f"{len(sorted_df[sorted_df['type']==key])}" for key, _ in sort_order.items())
    lines = [
        "System Generated by MLM \n",
        "1.0\n",
        f"{A1[0]} {A1[1]} 0.0\n",
        f"{A2[0]} {A2[1]} 0.0\n",
        f"{A3[0]} {A3[1]} {A3[2]}\n",
        f"{element_type}\n",
        f"{element_count} \n",
        "Cartesian\n"
    ]
    atomic_positions = atom_data[['x', 'y', 'z']]
    # Write the buffer to the file
    with open(filename, 'w') as file:
        file.writelines(lines)
        # Use to_csv for efficient DataFrame writing
        atomic_positions.to_csv(file, sep=' ', index=False, header=False, mode='a')

In [261]:
write_vasp_optimized(f"mos2_ttl_{theta1}_{theta2}.vasp",A1,A2,A3, sorted_df, sort_order)
print(f"Wrote === mos2_ttl_{theta1}_{theta2}.vasp")

Wrote === mos2_ttl_21.77_38.21000000000241.vasp


# LAMMPS Writer

In [59]:
selected_atoms_df.head()

,x,y,z,type,sort_key,numeric_id,mass
1218,-0.237450,1.416615,12.273334,Mo,0,2,95.940
1222,-0.237450,3.244506,10.696824,S,1,1,32.065
1223,-0.237450,3.244506,13.850458,S,1,3,32.065
1305,1.345549,4.158452,12.273334,Mo,0,2,95.940
1306,2.928549,3.244506,10.696824,S,1,1,32.065


In [57]:
len(selected_atoms_df)

42

## Set layer id as required by KC forcefield

In [58]:
unique_z = sorted(selected_atoms_df['z'].unique())

numeric_id_dict = {value: index+1 for index, value in enumerate(unique_z)}

print(f"Numeric Id: {numeric_id_dict}")

# Create a new column to map the sort order
selected_atoms_df['numeric_id'] = selected_atoms_df['z'].map(numeric_id_dict)




Numeric Id: {10.696823867: 1, 12.273333597: 2, 13.850457828: 3, 16.696823867: 4, 18.273333597: 5, 19.850457828: 6}


/tmp/ipykernel_784534/4183766918.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_atoms_df['numeric_id'] = selected_atoms_df['z'].map(numeric_id_dict)


## Set Masses Requires manual input

In [60]:
unique_element = selected_atoms_df['type'].unique()

print(f"Unique Elements: {unique_element}")



Unique Elements: ['Mo' 'S']


In [61]:
mass_type_dict = {'B': 10.811, 'C': 12.011, 'N': 14.007, 'O': 15.999, 
                  'S': 32.065, 'Mo': 95.94, 'W': 183.84, 'Se': 78.96, 
                  'Te': 127.6, 'Pb': 207.2, 'Bi': 208.980, 'Ti': 47.867}

In [62]:

selected_atoms_df['mass'] = selected_atoms_df['type'].map(mass_type_dict)

/tmp/ipykernel_784534/3153881743.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_atoms_df['mass'] = selected_atoms_df['type'].map(mass_type_dict)


In [63]:
selected_atoms_df.head()

,x,y,z,type,sort_key,numeric_id,mass
1218,-0.237450,1.416615,12.273334,Mo,0,2,95.940
1222,-0.237450,3.244506,10.696824,S,1,1,32.065
1223,-0.237450,3.244506,13.850458,S,1,3,32.065
1305,1.345549,4.158452,12.273334,Mo,0,2,95.940
1306,2.928549,3.244506,10.696824,S,1,1,32.065


In [64]:
sorted(selected_atoms_df['numeric_id'].unique())

[1, 2, 3, 4, 5, 6]

In [65]:
def write_lammps_data(filename, A1, A2, A3, atom_data, mass_type_dict):
    
    angle_A1 = angle.angle_between(A1,np.array([1,0,0]))
    angle_A2 = angle.angle_between(A2,np.array([1,0,0]))
    if angle_A1 < angle_A2:
        rot_rad = np.deg2rad(angle_A1)
    else:
        rot_rad = np.deg2rad(angle_A2)
    
    rotation_matrix = np.array([[np.cos(-rot_rad), -np.sin(-rot_rad), 0],
                            [np.sin(-rot_rad), np.cos(-rot_rad), 0],
                            [0, 0, 1]])

    A11 = np.round((rotation_matrix @ A1.T),4)
    A22 = np.round((rotation_matrix @ A2.T),4)
    
    type_counts = sorted(atom_data['numeric_id'].unique())
    lines = [
        "System Generated by MLM \n\n",
        f"{len(atom_data)} atoms\n",
        f"{int(atom_data['numeric_id'].nunique())} atom types\n\n",
        f"0.0 {max(A11[0],A22[0],A3[0])} xlo xhi\n"
        f"0.0 {max(A11[1],A22[1],A3[1])} ylo yhi\n",
        f"0.0 {max(A11[2],A22[2],A3[2])} zlo zhi\n",
        f"{sorted([A11[0],A22[0],A3[0]],reverse=True)[1]} 0.0 0.0 xy xz yz\n\n",
        "Masses\n\n",
    ]
    # Loop to print each element type and count 
    for key in type_counts:
        mass_value = atom_data.loc[atom_data['numeric_id'] == key, 'mass'].values[0] 
        lines.append(f"{key} {mass_value}\n") 
        
    lines.extend([ "\nAtoms #atomic \n\n" ])
    
    atomic_positions = atom_data[['x', 'y', 'z']].values
    rotated_atomic_pos = atomic_positions @ rotation_matrix.T
    rot_atomic_positions = pd.DataFrame(rotated_atomic_pos, columns=['x', 'y', 'z'])
    rot_atomic_positions.reset_index(inplace=True)
    rot_atomic_positions['id'] = rot_atomic_positions.index + 1
    rot_atomic_positions['numeric_id'] = atom_data['numeric_id'].values
    data = rot_atomic_positions[['id', 'numeric_id', 'x', 'y', 'z']]
    
    
    # Write the buffer to the file
    with open(filename, 'w') as file:
        file.writelines(lines)
        # Use to_csv for efficient DataFrame writing
        data.to_csv(file, sep=' ', index=False, header=False, mode='a')

In [66]:
write_lammps_data("lammps_test_mos2_21.8.data", A1, A2, A3, selected_atoms_df, mass_type_dict)

In [31]:
np.linalg.norm(np.array([-8.6355194156, -2.1367369111]))

np.float64(8.895945155207608)

In [32]:
np.linalg.norm(np.array([-2.4672912616, -8.5469476444]))

np.float64(8.895945155276218)

In [33]:
mlf.angle_between(np.array([-8.6355194156, -2.1367369111]),np.array([-2.4672912616, -8.5469476444]) )

60.000000000017